# 02 — Anomaly Detection

Exploration of unusual weather events using three detection methods:
- Isolation Forest
- Local Outlier Factor (LOF)
- Z-score threshold

Anomalies are flagged when at least 2 of 3 methods agree.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

In [ ]:
# Load processed daily global series
daily = pd.read_parquet('../data/processed/daily_global.parquet')
daily['date'] = pd.to_datetime(daily['date'])
print(f'Shape: {daily.shape}')
daily.head(3)

## Run anomaly detection

In [ ]:
from anomaly_detection import run_anomaly_detection, describe_anomalies
from feature_engineering import run_feature_engineering

feature_df = run_feature_engineering(daily)
df_annotated, summary = run_anomaly_detection(feature_df)
print(summary)

## Annotated time series

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df_annotated['date'], df_annotated['temperature_celsius'],
        color='steelblue', lw=1, alpha=0.7, label='Normal')

mask = df_annotated['anomaly']
ax.scatter(df_annotated['date'][mask], df_annotated['temperature_celsius'][mask],
           color='red', s=60, zorder=5, marker='x', label='Anomaly')

ax.set_title('Global Temperature with Anomaly Markers', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.legend()
plt.tight_layout()
plt.show()

## Method-by-method count comparison

In [ ]:
counts = {
    'Isolation Forest': summary['isolation_forest_count'],
    'LOF': summary['lof_count'],
    'Z-score': summary['zscore_count'],
    'Ensemble (>=2)': summary['ensemble_count'],
}

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(list(counts.keys()), list(counts.values()), color='steelblue', edgecolor='white')
ax.set_title('Anomaly Count by Detection Method', fontsize=12, fontweight='bold')
ax.set_xlabel('Number of Anomalies')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Anomaly event table

In [ ]:
events = describe_anomalies(df_annotated)
print(f'Total anomaly events: {len(events)}')
events.head(20)